# Spectra to ChemNet Encoder

We need to use an encoder to get ChemNet embeddings from the spectrum

## Processing

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

# from fcd_torch import FCD
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA

from fcd_torch import FCD
import rdkit

### Data Upload

Whats going on?
1. Read in the dataset, in this case df2

In [ ]:
# The 5/20 dataset with rat based toxicity data
df2 = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/MIT_LL_data2.csv")
print(df2.shape)
df2.head()

### Correct Dataframe processing

What's going on?
1. Ensure the ionization modes are all written identically and remove ionzation modes labels N/A.
2. Remove single quotation marks from whole dataset.

In [ ]:
# Uniformity of ionization model labels
print(df2["Ionization_Mode"].unique())
df2["Ionization_Mode"] = df2["Ionization_Mode"].replace("'Positive'", "'positive'")
print(df2["Ionization_Mode"].unique())

# Remove the N/A values in Ionization_Mode
df2 = df2[df2["Ionization_Mode"] != "'N/A'"]
print(df2["Ionization_Mode"].unique())

# Removing the '' from the SMILES
# Remove single quotes from all string columns in df2
df2 = df2.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)
#df2["SMILES_spectra"] = df2["SMILES_spectra"].str.replace("'", "")
df2.head()


### Data Split

What's going on?
1. Split the data by positive and negative inionization modes

In [ ]:
# Split into 2 sets, one for each ionization mode
df2_pos = df2[df2["Ionization_Mode"] == "positive"]
df2_neg = df2[df2["Ionization_Mode"] == "negative"]
print(df2_pos.shape)
print(df2_neg.shape)


### Spectra Visualizations

What's going on?
1. Singular spectra plot
2. Small grid of multiple spectra plots

In [ ]:
# Singular Spectrum Plot
spectrum = df2["Spectrum"].iloc[0]  # example: first row
pairs = spectrum.split()
x = [float(pair.split(":")[0].replace("'", "").replace('"', '')) for pair in pairs]
y = [float(pair.split(":")[1].replace("'", "").replace('"', '')) for pair in pairs]
#plt.plot(x, y, color = 'magenta',marker='o')
plt.bar(x,y, color = 'magenta', width = 0.8)
plt.xlabel("m/z")
plt.ylabel("Intensity")
plt.show()

In [ ]:
# Several Spectra Plots
fig, axes = plt.subplots(3, 3, figsize=(15, 8))
axes = axes.flatten()
colors = ['red', 'orange', 'yellow', 'green', 'blue', 'indigo', 'violet', 'cyan', 'magenta']  

for i in range(9):
    spectrum = df2["Spectrum"].iloc[i]
    pairs = spectrum.split()
    x = []
    y = []
    for pair in pairs:
        try:
            mz, intensity = pair.split(":")
            mz = float(mz.replace("'", "").replace('"', ''))
            intensity = float(intensity.replace("'", "").replace('"', ''))
            x.append(mz)
            y.append(intensity)
        except Exception:
            continue  # skip malformed pairs
    axes[i].bar(x, y, color=colors[i % len(colors)], width=0.8) # Bar plot
    #axes[i].plot(x, y, marker=',', color=colors[i % len(colors)]) # Line plot
    axes[i].set_title(f"Spectrum {i}")
    axes[i].set_xlabel("m/z")
    axes[i].set_ylabel("Intensity")

plt.tight_layout()
plt.show()

### Spectra data processing

What's going on?
1. Define function to convert spectrum from strings to matricies (pandas dataframes).
2. Excecute function on PIM.
3. Excecute function on NIM.

In [ ]:
def spectrum_string_to_dataframe(df, spectrum_col, smiles_col):
    """
    Converts a DataFrame with a spectrum column (string of 'x:y' pairs) into a matrix
    where columns are unique x values, rows are spectra (even for duplicate SMILES), and values are y (intensity).
    The index will match the original DataFrame.
    """
    # Collect all unique x values (m/z)
    x_values_set = set()
    spectra_list = []
    for idx, row in df.iterrows():
        spectrum = row[spectrum_col]
        pairs = spectrum.split()
        xys = []
        for pair in pairs:
            try:
                x, y = pair.split(":") # Split into x and y
                #x = float(x.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                #y = float(y.replace("'", "").replace('"', '')) # Remove quotes and convert to float (done in processing)
                xys.append((x, y))
                x_values_set.add(x)
            except Exception:
                continue
        spectra_list.append((row[smiles_col], dict(xys)))
    x_values = sorted(x_values_set) # Sort the x values to maintain order
    
    # Build the matrix
    matrix = []
    smiles_list = []
    for smiles, xy_dict in spectra_list:
        row = [xy_dict.get(x, 0.0) for x in x_values]
        matrix.append(row)
        smiles_list.append(smiles)
    df_matrix = pd.DataFrame(matrix, columns=[x for x in x_values]) # columns=[f"mz_{x}" for x in x_values]) to make stings
    df_matrix.insert(0, smiles_col, smiles_list)
    df_matrix.index = df.index  # preserve original row order/index
    return df_matrix

In [ ]:
# Positive ionization mode spectrum string to dataframe conversion
df2_pos_matrix = spectrum_string_to_dataframe(df2_pos,'Spectrum', 'SMILES_spectra')
print(df2_pos.shape) # (2214, 12)
print(df2_pos_matrix.shape) # (2214, 33385)
print(type(df2_pos_matrix))
df2_pos_matrix.head()


In [ ]:
# Negative ionization mode spectrum string to dataframe conversion
df2_neg_matrix = spectrum_string_to_dataframe(df2_neg,'Spectrum', 'SMILES_spectra')
print(df2_neg.shape) # (836, 12)
print(df2_neg_matrix.shape) # (836, 5887)
print(type(df2_neg_matrix))
df2_neg_matrix.head()


### Binning procedure preprocessing

What's going on?
1. Load in data in such a way that the kernel won't keep crashing.
2. Convert column names and all entries (aside from the SMILES labels) to floats and verify.
3. Examine maximum and minimum columns.
4. Sort dataframes.


In [ ]:
df2_pos_spectra = df2_pos_matrix
df2_neg_spectra = df2_neg_matrix

In [ ]:
# Assume df2_pos_spectra is your DataFrame
cols = df2_pos_spectra.columns.tolist()
# Keep the first column as is, convert the rest to float
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df2_pos_spectra.columns = new_cols

# Assume df2_neg_spectra is your DataFrame
cols = df2_neg_spectra.columns.tolist()
# Keep the first column as is, convert the rest to float
new_cols = [cols[0]] + [float(c) for c in cols[1:]]
df2_neg_spectra.columns = new_cols

In [ ]:
# Convert all elements except the first column to float
df2_pos_spectra.iloc[:, 1:] = df2_pos_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
df2_neg_spectra.iloc[:, 1:] = df2_neg_spectra.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

In [ ]:
all_float = all(isinstance(c, float) for c in df2_pos_spectra.columns[1:])
print("All columns are float:", all_float)
all_float = all(isinstance(c, float) for c in df2_neg_spectra.columns[1:])
print("All columns are float:", all_float)

In [ ]:
# Select all columns except the first
spectra = df2_pos_spectra.iloc[:, 1:]

# Check if every element is a float
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)

# Select all columns except the first
spectra = df2_neg_spectra.iloc[:, 1:]

# Check if every element is a float
all_float_elements = spectra.applymap(lambda x: isinstance(x, float)).all().all()
print("All elements are float:", all_float_elements)


In [ ]:
# Look at the column maximums
max_col_pos = max(df2_pos_spectra.columns[1:])
print("Largest column name (m/z):", max_col_pos)
max_col_neg = max(df2_neg_spectra.columns[1:])
print("Largest column name (m/z):", max_col_neg)

# Look at the column minimums
min_col_pos = min(df2_pos_spectra.columns[1:])
print("Smallest column name (m/z):", min_col_pos)
min_col_neg = min(df2_neg_spectra.columns[1:])
print("Smallest column name (m/z):", min_col_neg)

In [ ]:
# Here we will sort the columns so the m/z values are in ascending order
# Sort columns by float value, keep the first column (SMILES) first
# For negative ionization mode
cols = df2_neg_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df2_neg_spectra = df2_neg_spectra[sorted_cols]
# For positive ionization mode
cols = df2_pos_spectra.columns.tolist()
sorted_cols = [cols[0]] + sorted(cols[1:], key=float)
df2_pos_spectra = df2_pos_spectra[sorted_cols]

### Binning procedure execution

What's going on?
1. Save float converted, sorted dataframes as csv files and reload them in (problems with this as when reloaded the datadoesnt appear to be all floats anymore)
2. Define binning function: Bin by rounding columns to nearest integer and summing all element of columns binned together.
3. Excecute binning on PIM and verify
4. Excecute binning on NIM and verify

Note: This is the stage where I keep having issued of the Kernal dying when I try to run this code on my local machine.

In [ ]:
# Now we can save the dataframes to csv files
#df2_pos_spectra.to_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/df2_pos_spectra.csv", index=False)
# df2_neg_spectra.to_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/df2_neg_spectra.csv", index=False)
# Load the saved dataframes to check
# df2_pos_spectra = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/df2_pos_spectra.csv")
# df2_neg_spectra = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/df2_neg_spectra.csv")
print(df2_pos_spectra.shape)  # Should match df2_pos_spectra
print(df2_neg_spectra.shape)  # Should match df2_neg_spectra


In [ ]:
# Binning the spectra data 
def bin_spectra_by_integer_mz(df):
    """
    Bins the spectra data by rounding m/z (column names) to the nearest integer,
    then sums intensities for duplicate integer bins.
    Assumes first column is SMILES, rest are m/z columns (floats).
    """
    smiles_col = df.columns[0]
    spectra = df.iloc[:, 1:]
    # Map each column to its integer bin
    int_mz = [int(round(float(c))) for c in spectra.columns]
    spectra.columns = int_mz
    # Group columns by integer m/z and sum
    binned = spectra.groupby(level=0, axis=1).sum()
    # Add the SMILES column back
    binned.insert(0, smiles_col, df[smiles_col])
    return binned
    

In [ ]:
# Use the binning function on the positive ionization mode spectra
binned_pos = bin_spectra_by_integer_mz(df2_pos_spectra)


In [ ]:
# Verify the binning worked correctly
binned_pos.head()

In [ ]:
# Use the binning fuction on the negative ionization mode spectra
binned_neg = bin_spectra_by_integer_mz(df2_neg_spectra)

In [ ]:
# Verify the binning worked correctly
binned_neg.head()

In [ ]:
# It did work but there are many missing intgers so we will fill them with 0s
def fill_missing_integer_columns(df):
    """
    Ensures all integer columns from 1 to the maximum are present in the DataFrame.
    Missing columns are filled with zeros.
    Assumes the first column is the label (e.g., SMILES).
    """
    # Get the integer columns (skip the first column)
    int_cols = [col for col in df.columns[1:] if isinstance(col, int)]
    #min_col = min(int_cols)
    max_col = max(int_cols)
    all_int_cols = list(range(1, max_col + 1))
    # Find missing columns
    missing_cols = set(all_int_cols) - set(int_cols)
    # Add missing columns with zeros
    for col in missing_cols:
        df[col] = 0.0
    # Reorder columns: first column, then sorted integer columns
    ordered_cols = [df.columns[0]] + sorted(all_int_cols)
    df = df[ordered_cols]
    return df


In [ ]:
binned_pos_filled = fill_missing_integer_columns(binned_pos)

In [ ]:
binned_pos_filled.head()


In [ ]:
binned_neg_filled = fill_missing_integer_columns(binned_neg)

In [ ]:
binned_neg_filled.head()


In [ ]:
# Save the binned and filled dataframes to csv files
# binned_pos_filled.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/df2_binned_pos_filled.csv", index=False)
# binned_neg_filled.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/df2_binned_neg_filled.csv", index=False)
# Load the saved dataframes to check
df2_binned_pos_filled = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/df2_binned_pos_filled.csv")
df2_binned_neg_filled = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/df2_binned_neg_filled.csv")
print(df2_binned_pos_filled.shape)  # Should match binned_pos_filled
print(df2_binned_neg_filled.shape)  # Should match binned_neg_filled

## Quartile SMILES to ChemNet Projections

Currently this subsection only contains PCA projections of the 5/20/2025 dataset. Both with a quartile split and with a half and half split. The division ins't nearly as good as we saw with the 5/13/2025 dataset in both cases, but there may be some structure and expanding this to include other projections might be a good idea.

### Quartile Split

In [ ]:
# From df positive and negative we select on of each SMILES
df2_pos_unique = df2_pos.drop_duplicates(subset="SMILES_spectra")
print(df2_pos.shape)
print(df2_pos_unique.shape)
df2_neg_unique = df2_neg.drop_duplicates(subset="SMILES_spectra")
print(df2_neg.shape)
print(df2_neg_unique.shape)

In [ ]:
# Split positive unique by LD50 quartiles
q1_pos = df2_pos_unique["Response"].quantile(0.25)
q3_pos = df2_pos_unique["Response"].quantile(0.75)
bottom_pos = df2_pos_unique[df2_pos_unique["Response"] <= q1_pos].copy()
top_pos = df2_pos_unique[df2_pos_unique["Response"] >= q3_pos].copy()
bottom_pos["tox_quartile"] = 0
top_pos["tox_quartile"] = 1
df2_pos_quartiles = pd.concat([bottom_pos, top_pos])

# Split negative unique by LD50 quartiles
q1_neg = df2_neg_unique["Response"].quantile(0.25)
q3_neg = df2_neg_unique["Response"].quantile(0.75)
bottom_neg = df2_neg_unique[df2_neg_unique["Response"] <= q1_neg].copy()
top_neg = df2_neg_unique[df2_neg_unique["Response"] >= q3_neg].copy()
bottom_neg["tox_quartile"] = 0
top_neg["tox_quartile"] = 1
df2_neg_quartiles = pd.concat([bottom_neg, top_neg])

# Now df1_pos_quartiles and df1_neg_quartiles are your labeled DataFrames
print(df2_pos_quartiles.shape)
print(df2_neg_quartiles.shape)

In [ ]:
# Histogram of toxicity (full dataset)
plt.hist(df2["Response"].dropna(), bins=30, color='purple', edgecolor='black')
plt.xlabel("Response")
plt.ylabel("Count")
plt.title("Histogram of Response")
plt.show()

# Histogram of toxicity (positive dataset)
plt.hist(df2_pos_quartiles["Response"].dropna(), bins=30, color='blue', edgecolor='black')
plt.xlabel("Response")
plt.ylabel("Count")
plt.title("Histogram of Response (positive)")
plt.show()
# Histogram of toxicity (negative dataset)
plt.hist(df2_neg_quartiles["Response"].dropna(), bins=30, color='red', edgecolor='black')
plt.xlabel("Response")
plt.ylabel("Count")
plt.title("Histogram of Response (negative)")
plt.show()

In [ ]:
# Cate's smiles to ChemNet embedding code
def get_chemnet_emb_from_smiles(smiles_list):
    """
    Get ChemNet embeddings from a list of SMILES strings.

    Parameters:
    smiles_list (list): List of SMILES strings.

    Returns:
    dict: A dictionary mapping each SMILES string to its corresponding ChemNet embedding.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fcd = FCD(device, n_jobs=1)
    
    smiles_emb_dict = {}

    for smiles in smiles_list:
        try:
            emb = fcd.get_predictions([smiles])[0]
            smiles_emb_dict[smiles] = list(emb)
        except KeyError as e:
            if e == 'PropertyTable':
                smiles_emb_dict[smiles] = 'unknown'

    return smiles_emb_dict

In [ ]:
# Generate dictionaries of ChemNet embeddings for the positive and negative datasets
ChemNet_of_df2neg_quartiles = get_chemnet_emb_from_smiles(df2_neg_quartiles["SMILES_spectra"])
ChemNet_of_df2pos_quartiles = get_chemnet_emb_from_smiles(df2_pos_quartiles["SMILES_spectra"])

In [ ]:
# Dataframe of positive ionization mode ChemNet embeddings
# Convert the embedding dictionary to a DataFrame
emb_df = pd.DataFrame.from_dict(ChemNet_of_df2pos_quartiles, orient='index')
emb_df.index.name = 'SMILES_spectra'
emb_df.reset_index(inplace=True)

# Merge with the toxicity quartile column (no one-hot encoding)
chemnet_emb_df2pos = emb_df.merge(
    df2_pos_quartiles[['SMILES_spectra', 'tox_quartile']],
    on='SMILES_spectra',
    how='left'
)

# Ensure the last column is 0 or 1
chemnet_emb_df2pos['tox_quartile'] = chemnet_emb_df2pos['tox_quartile'].astype(int)

print(chemnet_emb_df2pos.shape)  # (n, 514)
#chemnet_emb_df2pos.head()

In [ ]:
# Dataframe of negative ionization mode ChemNet embeddings
# Convert the embedding dictionary to a DataFrame
emb_df = pd.DataFrame.from_dict(ChemNet_of_df2neg_quartiles, orient='index')
emb_df.index.name = 'SMILES_spectra'
emb_df.reset_index(inplace=True)

# Merge with the toxicity quartile column (no one-hot encoding)
chemnet_emb_df2neg = emb_df.merge(
    df2_neg_quartiles[['SMILES_spectra', 'tox_quartile']],
    on='SMILES_spectra',
    how='left'
)

# Ensure the last column is 0 or 1
chemnet_emb_df2neg['tox_quartile'] = chemnet_emb_df2neg['tox_quartile'].astype(int)

print(chemnet_emb_df2neg.shape)  # Should be (n, 514)
#chemnet_emb_df2neg.head()

In [ ]:
# Select only the embedding columns (exclude SMILES and tox_quartile)
X_pos = chemnet_emb_df2pos.drop(columns=['SMILES_spectra', 'tox_quartile']).values
y_pos = chemnet_emb_df2pos['tox_quartile'].values

X_neg = chemnet_emb_df2neg.drop(columns=['SMILES_spectra', 'tox_quartile']).values
y_neg = chemnet_emb_df2neg['tox_quartile'].values

# PCA to 3 components
pca3 = PCA(n_components=3)
X_pos_pca3 = pca3.fit_transform(X_pos)
X_neg_pca3 = pca3.fit_transform(X_neg)

# PCA to 2 components
pca2 = PCA(n_components=2)
X_pos_pca2 = pca2.fit_transform(X_pos)
X_neg_pca2 = pca2.fit_transform(X_neg)

# Plot 3D PCA for positive
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
colors = ['blue' if t == 0 else 'red' for t in y_pos]
ax.scatter(X_pos_pca3[:,0], X_pos_pca3[:,1], X_pos_pca3[:,2], c=colors, alpha=0.7)
ax.set_title('Positive Ionization Mode: 3D PCA')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
red_patch = plt.Line2D([0], [0], marker='o', color='w', label='High Toxicity', markerfacecolor='red', markersize=10)
blue_patch = plt.Line2D([0], [0], marker='o', color='w', label='Low Toxicity', markerfacecolor='blue', markersize=10)
ax.legend(handles=[red_patch, blue_patch])
plt.show()

# Plot 2D PCA for positive
plt.figure(figsize=(8,6))
plt.scatter(X_pos_pca2[:,0], X_pos_pca2[:,1], c=colors, alpha=0.7)
plt.title('Positive Ionization Mode: 2D PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(handles=[red_patch, blue_patch])
plt.show()

# Plot 3D PCA for negative
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
colors = ['blue' if t == 0 else 'red' for t in y_neg]
ax.scatter(X_neg_pca3[:,0], X_neg_pca3[:,1], X_neg_pca3[:,2], c=colors, alpha=0.7)
ax.set_title('Negative Ionization Mode: 3D PCA')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
plt.legend(handles=[red_patch, blue_patch])
plt.show()

# Plot 2D PCA for negative
plt.figure(figsize=(8,6))
plt.scatter(X_neg_pca2[:,0], X_neg_pca2[:,1], c=colors, alpha=0.7)
plt.title('Negative Ionization Mode: 2D PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(handles=[red_patch, blue_patch])
plt.show()

### Halves Split

In [ ]:
### Data split, no removal but differentiation of above of below half toxicity
# Split positive unique by LD50 halves
h1_pos = df2_pos_unique["Response"].quantile(0.5)
bottom_half_pos = df2_pos_unique[df2_pos_unique["Response"] <= h1_pos].copy()
top_half_pos = df2_pos_unique[df2_pos_unique["Response"] > h1_pos].copy()
bottom_half_pos["tox_half"] = 0
top_half_pos["tox_half"] = 1
df2_pos_halves = pd.concat([bottom_half_pos, top_half_pos])

# Split negative unique by LD50 halves (fixed)
h1_neg = df2_neg_unique["Response"].quantile(0.5)
bottom_half_neg = df2_neg_unique[df2_neg_unique["Response"] <= h1_neg].copy()
top_half_neg = df2_neg_unique[df2_neg_unique["Response"] > h1_neg].copy()
bottom_half_neg["tox_half"] = 0
top_half_neg["tox_half"] = 1
df2_neg_halves = pd.concat([bottom_half_neg, top_half_neg])

print(df2_pos_halves.shape)
print(df2_neg_halves.shape)

In [ ]:
### Get the ChemNet embeddings for the halves
# Generate dictionaries of ChemNet embeddings for the positive and negative datasets
ChemNet_of_df2neg_halves = get_chemnet_emb_from_smiles(df2_neg_halves["SMILES_spectra"])
ChemNet_of_df2pos_halves = get_chemnet_emb_from_smiles(df2_pos_halves["SMILES_spectra"])

In [ ]:
# Dataframe of positive ionization mode ChemNet embeddings
# Convert the embedding dictionary to a DataFrame
emb_df_full = pd.DataFrame.from_dict(ChemNet_of_df2pos_halves, orient='index')
emb_df_full.index.name = 'SMILES_spectra'
emb_df_full.reset_index(inplace=True)

# Merge with the toxicity quartile column (no one-hot encoding)
chemnet_emb_df2pos_full = emb_df_full.merge(
    df2_pos_halves[['SMILES_spectra', 'tox_half']],
    on='SMILES_spectra',
    how='left'
)

# Ensure the last column is 0 or 1
chemnet_emb_df2pos_full['tox_half'] = chemnet_emb_df2pos_full['tox_half'].astype(int)

print(chemnet_emb_df2pos_full.shape)  # Should be (n, 514)
#chemnet_emb_df2pos_full.head()

In [ ]:
# Dataframe of negative ionization mode ChemNet embeddings
# Convert the embedding dictionary to a DataFrame
emb_df_full = pd.DataFrame.from_dict(ChemNet_of_df2neg_halves, orient='index')
emb_df_full.index.name = 'SMILES_spectra'
emb_df_full.reset_index(inplace=True)

# Merge with the toxicity quartile column (no one-hot encoding)
chemnet_emb_df2neg_full = emb_df_full.merge(
    df2_neg_halves[['SMILES_spectra', 'tox_half']],
    on='SMILES_spectra',
    how='left'
)

# Ensure the last column is 0 or 1
chemnet_emb_df2neg_full['tox_half'] = chemnet_emb_df2neg_full['tox_half'].astype(int)

print(chemnet_emb_df2neg_full.shape)  # Should be (n, 514)
#chemnet_emb_df2neg_full.head()

In [ ]:
### PCA Implementation
# Select only the embedding columns (exclude SMILES and tox_quartile)
X_pos = chemnet_emb_df2pos_full.drop(columns=['SMILES_spectra', 'tox_half']).values
y_pos = chemnet_emb_df2pos_full['tox_half'].values

X_neg = chemnet_emb_df2neg_full.drop(columns=['SMILES_spectra', 'tox_half']).values
y_neg = chemnet_emb_df2neg_full['tox_half'].values

# PCA to 3 components
pca3 = PCA(n_components=3)
X_pos_pca3 = pca3.fit_transform(X_pos)
X_neg_pca3 = pca3.fit_transform(X_neg)

# PCA to 2 components
pca2 = PCA(n_components=2)
X_pos_pca2 = pca2.fit_transform(X_pos)
X_neg_pca2 = pca2.fit_transform(X_neg)

### PCA Plots
# Plot 3D PCA for positive
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
colors = ['blue' if t == 0 else 'red' for t in y_pos]
ax.scatter(X_pos_pca3[:,0], X_pos_pca3[:,1], X_pos_pca3[:,2], c=colors, alpha=0.7)
ax.set_title('Positive Ionization Mode: 3D PCA (Full Dataset)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
red_patch = plt.Line2D([0], [0], marker='o', color='w', label='High Toxicity', markerfacecolor='red', markersize=10)
blue_patch = plt.Line2D([0], [0], marker='o', color='w', label='Low Toxicity', markerfacecolor='blue', markersize=10)
ax.legend(handles=[red_patch, blue_patch])
plt.show()

# Plot 2D PCA for positive
plt.figure(figsize=(8,6))
plt.scatter(X_pos_pca2[:,0], X_pos_pca2[:,1], c=colors, alpha=0.7)
plt.title('Positive Ionization Mode: 2D PCA (Full Dataset)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(handles=[red_patch, blue_patch])
plt.show()

# Plot 3D PCA for negative
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
colors = ['blue' if t == 0 else 'red' for t in y_neg]
ax.scatter(X_neg_pca3[:,0], X_neg_pca3[:,1], X_neg_pca3[:,2], c=colors, alpha=0.7)
ax.set_title('Negative Ionization Mode: 3D PCA (Full Dataset)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.legend(handles=[red_patch, blue_patch])
plt.show()

# Plot 2D PCA for negative
plt.figure(figsize=(8,6))
plt.scatter(X_neg_pca2[:,0], X_neg_pca2[:,1], c=colors, alpha=0.7)
plt.title('Negative Ionization Mode: 2D PCA (Full Dataset)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(handles=[red_patch, blue_patch])
plt.show()

## Encoder

We have the data in a useful form, 2 dataframes with float and x values as the column names and the y values as columns entries. We need to find some binning method to make this more useful to us in hitting it with an encoder.

The sizes of the two dataframes that we are currently dealing with are (2214, 33385) for the postive matrix and (836, 5887) for the negative matrix. This is prior to any binning strategy.

Ways to make the endocde adaptable:
- Set the input dimesion as a varaible and the layers as functions of that variable (consider that we cant have decimals)
- Set up several encoders, and make a encoder selection method. 

## Data processing

### ChemNet Spectra from SMILES with repeats

In [ ]:
# # Redefine the fuction so it makes a list rather than a dictionary, Done to get dataset
# def get_chemnet_emb_from_smiles_list(smiles_list):
#     """
#     Get ChemNet embeddings for a list of SMILES strings, preserving order and duplicates.
#     Returns a list of embeddings (or 'unknown') in the same order as input.
#     """
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     fcd = FCD(device, n_jobs=1)
#     embeddings = []
#     for smiles in smiles_list:
#         try:
#             emb = fcd.get_predictions([smiles])[0]
#             embeddings.append(list(emb))
#         except KeyError as e:
#             if e == 'PropertyTable':
#                 embeddings.append('unknown')
#     return embeddings

# # For positive mode
# smiles_list_pos = df2_pos['SMILES_spectra'].tolist()
# embeddings_pos = get_chemnet_emb_from_smiles_list(smiles_list_pos)
# chemnet_emb_df2pos_validation = pd.DataFrame(embeddings_pos)
# chemnet_emb_df2pos_validation.insert(0, 'SMILES_spectra', smiles_list_pos)
# print(chemnet_emb_df2pos_validation.shape)  # Should be (2214, n_features+1)

# # For negative mode
# smiles_list_neg = df2_neg['SMILES_spectra'].tolist()
# embeddings_neg = get_chemnet_emb_from_smiles_list(smiles_list_neg)
# chemnet_emb_df2neg_validation = pd.DataFrame(embeddings_neg)
# chemnet_emb_df2neg_validation.insert(0, 'SMILES_spectra', smiles_list_neg)
# print(chemnet_emb_df2neg_validation.shape)  # Should be (836, n_features+1)

In [ ]:
# # Save the ChemNet embeddings DataFrames to CSV files

# chemnet_emb_df2pos_validation.to_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_emb_df2pos_validation.csv", index=False)
# chemnet_emb_df2neg_validation.to_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_emb_df2neg_validation.csv", index=False)

# print(chemnet_emb_df2pos_validation.shape)
# print(chemnet_emb_df2neg_validation.shape)

In [ ]:
# # What constitutes training and validation sets?

# # Load the ChemNet embeddings DataFrames from CSV files: Contaions the SMILES and the embeddings of each chemical
# # This is the validation set
# chemnet_emb_df2pos_validation = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_emb_df2pos_validation.csv")
# chemnet_emb_df2neg_validation = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/chemnet_emb_df2neg_validation.csv")
# print(chemnet_emb_df2pos_validation.shape)
# print(chemnet_emb_df2neg_validation.shape)

# # Load the spectra dataframes
# # This is the training set
# df2_pos_spectra = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/df2_pos_spectra.csv")
# df2_neg_spectra = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/df2_neg_spectra.csv")
# print(df2_pos_spectra.shape)
# print(df2_neg_spectra.shape)

In [ ]:
print(df2_pos_spectra.shape)
print(df2_neg_spectra.shape)

### ChemNet from SMILES without repeats

In [ ]:
# Cate's smiles to ChemNet embedding code
def get_chemnet_emb_from_smiles(smiles_list):
    """
    Get ChemNet embeddings from a list of SMILES strings.

    Parameters:
    smiles_list (list): List of SMILES strings.

    Returns:
    dict: A dictionary mapping each SMILES string to its corresponding ChemNet embedding.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    fcd = FCD(device, n_jobs=1)
    
    smiles_emb_dict = {}

    for smiles in smiles_list:
        try:
            emb = fcd.get_predictions([smiles])[0]
            smiles_emb_dict[smiles] = list(emb)
        except KeyError as e:
            if e == 'PropertyTable':
                smiles_emb_dict[smiles] = 'unknown'

    return smiles_emb_dict

In [ ]:
# Get the ChemNet embeddings for the SMILES in df2_pos and df2_neg then save them as dataframes
# Generate dictionaries of ChemNet embeddings for the positive and negative datasets
ChemNet_of_df2pos_no_repeats_dict = get_chemnet_emb_from_smiles(df2_pos["SMILES_spectra"])
ChemNet_of_df2neg_no_repeats_dict = get_chemnet_emb_from_smiles(df2_neg["SMILES_spectra"])


In [ ]:
# Save both as dataframes
ChemNet_of_df2pos_no_repeats = pd.DataFrame.from_dict(ChemNet_of_df2pos_no_repeats_dict, orient='index')
ChemNet_of_df2pos_no_repeats.reset_index(inplace=True)
ChemNet_of_df2pos_no_repeats.rename(columns={'index': 'SMILES'}, inplace=True)
ChemNet_of_df2neg_no_repeats = pd.DataFrame.from_dict(ChemNet_of_df2neg_no_repeats_dict, orient='index')
ChemNet_of_df2neg_no_repeats.reset_index(inplace=True)
ChemNet_of_df2neg_no_repeats.rename(columns={'index': 'SMILES'}, inplace=True)
# Print the Shapes
print(ChemNet_of_df2pos_no_repeats.shape)  # Should be (n, 513)
print(ChemNet_of_df2neg_no_repeats.shape)  # Should be (n, 513)

In [ ]:
# # Saves the ChemNet_of_df2pos_no_repeats and ChemNet_of_df2neg_no_repeats as new dataframes
# ChemNet_of_df2pos_no_repeats.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df2pos_no_repeats.csv", index=False)
# ChemNet_of_df2neg_no_repeats.to_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df2neg_no_repeats.csv", index=False)

# Load the ChemNet embeddings DataFrames from CSV files
ChemNet_of_df2pos_no_repeats = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df2pos_no_repeats.csv")
ChemNet_of_df2neg_no_repeats = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df2neg_no_repeats.csv")

# Print the shapes to verify
print(ChemNet_of_df2pos_no_repeats.shape)  # Should be (n, 513)
print(ChemNet_of_df2neg_no_repeats.shape)  # Should be (n, 513)

## Refining of needed functions:


In [ ]:
def create_dataset_tensors(spectra_dataset, embedding_df, device, start_idx=None, stop_idx=None):
    """
    Create tensors from the provided spectra dataset and embedding DataFrame.

    Parameters:
    ----------
    spectra_dataset : pd.DataFrame
        DataFrame containing spectral data and chemical labels. Assumes specific 
        columns for processing based on the `carl` flag.

    embedding_df : pd.DataFrame
        DataFrame containing embeddings for chemicals, with 'Embedding Floats' 
        column corresponding to ChemNet embeddings.

    device : torch.device
        The device (CPU or GPU) on which to store the tensors.

    carl : bool, optional
        If True, processes the dataset assuming it has a different structure 
        (specifically without an 'Unnamed: 0' column). Default is False.

    Returns:
    -------
    tuple
        A tuple containing:
        - embeddings_tensor (torch.Tensor): Tensor of true embeddings for the chemicals.
        - spectra_tensor (torch.Tensor): Tensor of spectral data.
        - chem_encodings_tensor (torch.Tensor): Tensor of chemical name encodings.
        - spectra_indices_tensor (torch.Tensor): Tensor of indices corresponding to the spectra.
    """
    spectra = spectra_dataset.iloc[:,1:-1]

    #chem_encodings = spectra_dataset.iloc[:,-8:]

    # create tensors of spectra, true embeddings, and chemical name encodings for train and val
    chem_labels = list(spectra_dataset['SMILES_spectra'])
    embeddings_tensor = torch.Tensor([embedding_df.loc[embedding_df['SMILES'] == chem_name].iloc[0, 1:].values.astype(float) for chem_name in chem_labels]).to(device)
    spectra_tensor = torch.Tensor(spectra.values).to(device)
    #chem_encodings_tensor = torch.Tensor(chem_encodings.values).to(device)
    spectra_indices_tensor = torch.Tensor(spectra_dataset['index'].to_numpy()).to(device)

    return embeddings_tensor, spectra_tensor, spectra_indices_tensor #, chem_encodings_tensor


## Encoder implementation

If we want to actaully train an encoder then we will need to make a split of the data. We can only use the spectra with over 5 elements and make a train test split such that for each SMILES there are some training and some testing spectra for each unique smiles. 

In [ ]:
# # Training and validation dataset split for the positive ionization mode
# pos_data = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/df2_pos_spectra.csv")
# # Count occurrences of each SMILES_spectra
# counts = pos_data['SMILES_spectra'].value_counts()
# # Keep only SMILES_spectra with at least 5 entries
# valid_smiles = counts[counts >= 5].index
# filtered_pos_data = pos_data[pos_data['SMILES_spectra'].isin(valid_smiles)].copy()

# from sklearn.model_selection import train_test_split

# train_indices = []
# test_indices = []

# for smiles, group in filtered_pos_data.groupby('SMILES_spectra'):
#     idx = group.index.tolist()
#     # Ensure at least 2 in each set
#     if len(idx) < 4:
#         continue  # skip if not enough to split 2/2
#     test_idx = np.random.choice(idx, size=2, replace=False)
#     train_idx = list(set(idx) - set(test_idx))
#     # If more than 4, split the rest randomly
#     if len(train_idx) > 2:
#         train_idx, extra_test_idx = train_test_split(train_idx, test_size=2, random_state=42)
#         test_idx = np.concatenate([test_idx, extra_test_idx])
#     train_indices.extend(train_idx)
#     test_indices.extend(test_idx)

# train_data = filtered_pos_data.loc[train_indices].reset_index(drop=True)
# test_data = filtered_pos_data.loc[test_indices].reset_index(drop=True)

# print(train_data.shape)
# print(test_data.shape)

In [ ]:
# Training and validation dataset split for the positive ionization mode
pos_data = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/df2_binned_pos_filled.csv")
# Count occurrences of each SMILES_spectra
counts = pos_data['SMILES_spectra'].value_counts()
# Keep only SMILES_spectra with at least 4 entries
valid_smiles = counts[counts >= 4].index
filtered_pos_data = pos_data[pos_data['SMILES_spectra'].isin(valid_smiles)].copy()

train_indices = []
test_indices = []

for smiles, group in filtered_pos_data.groupby('SMILES_spectra'):
    idx = group.index.tolist()
    n = len(idx)
    np.random.shuffle(idx)  # shuffle for randomness
    split = n // 2
    test_idx = idx[:split]
    train_idx = idx[split:]
    # If odd, train gets the extra
    train_indices.extend(train_idx)
    test_indices.extend(test_idx)

train_data = filtered_pos_data.loc[train_indices].reset_index(drop=True)
test_data = filtered_pos_data.loc[test_indices].reset_index(drop=True)

# Add an 'index' column for downstream compatibility
train_data['index'] = train_data.index
test_data['index'] = test_data.index

print(train_data.shape)
print(test_data.shape)


In [ ]:
train_data.head()
print(type(train_data))
train_data_copy = train_data.copy()
print(type(train_data_copy))

In [ ]:
print(train_data['SMILES_spectra'].value_counts())

In [ ]:
# Input the Encoder file
#%% 
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import functions_enc as f

#%% 
# This cell needs to be updated with your own paths/requirements
batch_size = 64

# Train and validation datasets are determined by the above code
train_data = train_data
# train_data = pd.read_csv("/") 
val_data = test_data
# val_data = pd.read_csv(" ") 

# Chemical names and ChemNet embeddings
name_smiles_embedding_df = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df2pos_no_repeats.csv")

epochs=1000
lr=0.0001
criterion=nn.MSELoss()
output_size = 512
num_layers = 10

In [ ]:
#name_smiles_embedding_df.head()
#print(name_smiles_embedding_df['SMILES'].value_counts())

In [ ]:
#%%
# Encoder architecture (With Validation Set)
class Encoder(nn.Module):
    def __init__(self, input_size, output_size, num_layers):
        super().__init__()
        layers = []
        layer_sizes = np.linspace(input_size, output_size, num_layers + 1, dtype=int)
        for i in range(num_layers):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < num_layers - 1:
                layers.append(nn.LeakyReLU(inplace=True))
        self.encoder = nn.Sequential(*layers)

    def forward(self, x):
        return self.encoder(x)

def train_model(model, train_data, val_data, epochs, learning_rate, criterion, device):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch, true_embeddings, _ in train_data:
            batch = batch.to(device)
            true_embeddings = true_embeddings.to(device)

            optimizer.zero_grad()
            batch_predicted_embeddings = model(batch)
            loss = criterion(batch_predicted_embeddings, true_embeddings)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        average_train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_batch, val_true_embeddings, _ in val_data:
                val_batch = val_batch.to(device)
                val_true_embeddings = val_true_embeddings.to(device)

                val_batch_predicted_embeddings = model(val_batch)

                val_loss = criterion(val_batch_predicted_embeddings, val_true_embeddings)
                val_loss += loss.item()
        average_val_loss = val_loss / len(val_loader)

        print(f'Epoch [{epoch+1}/{epochs}]')
        print(f'   Training loss: {average_train_loss}')
        print(f'   Validation loss: {average_val_loss}')

    return model
#%%


In [ ]:
# # Encoder architecture (Without Validation Set)
# class Encoder(nn.Module):
#     def __init__(self, input_size, output_size, num_layers):
#         super().__init__()
#         layers = []
#         layer_sizes = np.linspace(input_size, output_size, num_layers + 1, dtype=int)
#         for i in range(num_layers):
#             layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
#             if i < num_layers - 1:
#                 layers.append(nn.LeakyReLU(inplace=True))
#         self.encoder = nn.Sequential(*layers)

#     def forward(self, x):
#         return self.encoder(x)
# def train_model(model, train_data, epochs, learning_rate, criterion, device):
#     optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

#     for epoch in range(epochs):
#         model.train()
#         running_loss = 0.0
#         for batch, name_encodings, true_embeddings, _ in train_data:
#             batch = batch.to(device)
#             name_encodings = name_encodings.to(device)
#             true_embeddings = true_embeddings.to(device)

#             optimizer.zero_grad()
#             batch_predicted_embeddings = model(batch)
#             loss = criterion(batch_predicted_embeddings, true_embeddings)
#             loss.backward()
#             optimizer.step()
#             running_loss += loss.item()
#         average_train_loss = running_loss / len(train_data)

#         print(f'Epoch [{epoch+1}/{epochs}]')
#         print(f'   Training loss: {average_train_loss}')

#     return model

In [ ]:
# Encoder training
device = f.set_up_gpu()
# device = torch.device('cpu')

# Training set
y_train, x_train, train_indices_tensor = create_dataset_tensors(
    train_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
#sorted_chem_names = list(train_data.columns[-8:])
del train_data

# Validation set
y_val, x_val, val_indices_tensor = create_dataset_tensors(
    val_data, name_smiles_embedding_df, device, start_idx=2, stop_idx=-0)
del val_data

train_data = TensorDataset(x_train, y_train, train_indices_tensor)
val_data = TensorDataset(x_val, y_val, val_indices_tensor)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
#%%
encoder = Encoder(input_size=x_train.shape[1], output_size=output_size, num_layers=num_layers).to(device)

#%%
model = train_model(
    model=encoder,
    train_data=train_loader,
    val_data=val_loader,
    epochs=epochs,
    learning_rate=lr,
    criterion=criterion,
    device=device
)

In [ ]:
encoder.eval()
with torch.no_grad():
    test_output = encoder(x_val)  # output.shape will be (batch_size, 512)

with torch.no_grad():
    train_output = encoder(x_train)  # output.shape will be (batch_size, 512)

## PCA plots of encoder outputs

### PCA data processing

In [ ]:
# Mapping from SMILES to Response
smiles_to_response = df2_pos.drop_duplicates(subset='SMILES_spectra').set_index('SMILES_spectra')['Response']

# Ouptput dataframe
test_output_np = test_output.cpu().numpy()
test_output_df = pd.DataFrame(test_output_np)
test_output_df['SMILES_spectra'] = test_data['SMILES_spectra'].values  # 1:1 with test_data
cols = test_output_df.columns.tolist()
test_output_df = test_output_df[['SMILES_spectra'] + [c for c in cols if c != 'SMILES_spectra']]
test_output_df['Response'] = test_output_df['SMILES_spectra'].map(smiles_to_response)

# Check the output
test_output_df.head()
print(test_output_df.shape)
print(test_output_df['SMILES_spectra'].value_counts())

In [ ]:
# Find SMILES_spectra with 'occurances' or fewer occurrences
occurances = 20 # This can be changed
smiles_counts = test_output_df['SMILES_spectra'].value_counts()

#fewer_select_smiles = smiles_counts[smiles_counts <= occurances].index
more_select_smiles = smiles_counts[smiles_counts >= occurances].index
# Subset test_output_df
#test_output_df_sub = test_output_df[output_df['SMILES_spectra'].isin(fewer_select_smiles)].copy()
test_output_df_sub = test_output_df[test_output_df['SMILES_spectra'].isin(more_select_smiles)].copy()

print(test_output_df_sub.shape)
test_output_df_sub.head()

num_unique_smiles = test_output_df_sub['SMILES_spectra'].nunique()
print("Number of unique SMILES_spectra:", num_unique_smiles)



In [ ]:
test_output_df_sub.head(20)

In [ ]:
# Upload the Chemnet embeddings dataframe to provide true embeddings 
ChemNet_of_df2pos_no_repeats = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df2pos_no_repeats.csv")

# Get the unique SMILES_spectra for the subset
sub_smiles_set = set(test_output_df_sub['SMILES_spectra'].unique()) # With a unique dataframe this isn't strictly necessary

# Subset ChemNet_of_df2pos_no_repeats to only those SMILES
chemnet_sub = ChemNet_of_df2pos_no_repeats[ChemNet_of_df2pos_no_repeats['SMILES'].isin(sub_smiles_set)].copy()

# Check
print(chemnet_sub.shape)
chemnet_sub.head()


In [ ]:
# Merge with the Response column from df2 to get toxicity values
chemnet_sub = chemnet_sub.merge(
    df2[['SMILES_spectra', 'Response']],
    left_on='SMILES',
    right_on='SMILES_spectra',
    how='left'
)
# Remove the SMILES_spectra column after merging as it's redundant
chemnet_sub = chemnet_sub.drop(columns=['SMILES_spectra'])

# Keep only one row per unique SMILES (first occurrence)
chemnet_sub= chemnet_sub.drop_duplicates(subset='SMILES').copy()

print(chemnet_sub.shape)
chemnet_sub.head()



In [ ]:
# Create tox_epa12 column based on Response
test_output_df_sub['tox_epa12'] = (test_output_df_sub['Response'] <= 500).astype(int)
chemnet_sub['tox_epa12'] = (chemnet_sub['Response'] <= 500).astype(int)

# Check the disribution
print(test_output_df_sub['tox_epa12'].value_counts())
print(chemnet_sub['tox_epa12'].value_counts())


In [ ]:
print(test_output_df_sub.columns)

### PCA Plotting

In [ ]:
# Functions in plotting procedure
import matplotlib.cm as cm

# Isolate the embedding columns
def get_embedding_columns(df):
    # Select all columns except the first and last by index
    return df.columns[1:-2].tolist()

# Outlier removal function after PCA
def remove_outliers_on_axes(df, pc1_col, pc2_col, n_low_pc1=0, n_high_pc1=0, n_low_pc2=0, n_high_pc2=0):
    # Remove outliers on PC2
    df = df.sort_values(pc2_col, ascending=True)
    if n_low_pc2 > 0:
        df = df.iloc[n_low_pc2:]
    if n_high_pc2 > 0:
        df = df.iloc[:-n_high_pc2]
    # Remove outliers on PC1
    df = df.sort_values(pc1_col, ascending=True)
    if n_low_pc1 > 0:
        df = df.iloc[n_low_pc1:]
    if n_high_pc1 > 0:
        df = df.iloc[:-n_high_pc1]
    return df


# Outlier removal function before PCA
def remove_outliers_by_column(df, col, n_low=0, n_high=0):
    df_sorted = df.sort_values(col, ascending=True)
    if n_high > 0:
        df_sorted = df_sorted.iloc[:-n_high]
    if n_low > 0:
        df_sorted = df_sorted.iloc[n_low:]
    return df_sorted

In [ ]:
# PCA plot, color by toxicity

# Prepare data
emb_cols_output = get_embedding_columns(test_output_df_sub)
emb_cols_chemnet = get_embedding_columns(chemnet_sub)

# Get the actual data as numpy arrays
X_test_output = test_output_df_sub[emb_cols_output].values
X_chemnet = chemnet_sub[emb_cols_chemnet].values

# Combine for PCA fit
X_all = np.vstack([X_test_output, X_chemnet])
pca = PCA(n_components=2)
X_all_pca = pca.fit_transform(X_all)

# Re-split
X_test_output_pca = X_all_pca[:len(test_output_df_sub)]
X_chemnet_pca = X_all_pca[len(test_output_df_sub):]

# Plot
plt.figure(figsize=(8,6))

# Plot points and color by toxicity 
# Output
mask_test_output_red = test_output_df_sub['tox_epa12'] == 1
mask_test_output_blue = test_output_df_sub['tox_epa12'] == 0
plt.scatter(X_test_output_pca[mask_test_output_blue,0], X_test_output_pca[mask_test_output_blue,1], 
            c='blue', marker='x', label='test_output_df_sub, tox_epa12=0', alpha=0.7)
plt.scatter(X_test_output_pca[mask_test_output_red,0], X_test_output_pca[mask_test_output_red,1], 
            c='red', marker='x', label='test_output_df_sub, tox_epa12=1', alpha=0.7)

# ChemNet
mask_chemnet_red = chemnet_sub['tox_epa12'] == 1
mask_chemnet_blue = chemnet_sub['tox_epa12'] == 0
plt.scatter(X_chemnet_pca[mask_chemnet_blue,0], X_chemnet_pca[mask_chemnet_blue,1], 
            c='blue', marker='o', label='chemnet_sub, tox_epa12=0', alpha=0.7)
plt.scatter(X_chemnet_pca[mask_chemnet_red,0], X_chemnet_pca[mask_chemnet_red,1], 
            c='red', marker='o', label='chemnet_sub, tox_epa12=1', alpha=0.7)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA of ChemNet Embeddings')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PCA plot color by SMILES

# Prepare data
emb_cols_output = get_embedding_columns(test_output_df_sub)
emb_cols_chemnet = get_embedding_columns(chemnet_sub)

# Get the actual data as numpy arrays
X_test_output = test_output_df_sub[emb_cols_output].values
X_chemnet = chemnet_sub[emb_cols_chemnet].values

# Combine for PCA fit
X_all = np.vstack([X_test_output, X_chemnet])
pca = PCA(n_components=2)
X_all_pca = pca.fit_transform(X_all)

# Re-split
X_test_output_pca = X_all_pca[:len(test_output_df_sub)]
X_chemnet_pca = X_all_pca[len(test_output_df_sub):]

# Plot
plt.figure(figsize=(10,7))

# Plot points and color by SMILES 
# Output
unique_smiles_test_output = test_output_df_sub['SMILES_spectra'].unique()
color_map_test_output = {smiles: cm.tab20(i % 20) for i, smiles in enumerate(unique_smiles_test_output)}
test_output_colors = test_output_df_sub['SMILES_spectra'].map(color_map_test_output)
plt.scatter(X_test_output_pca[:,0], X_test_output_pca[:,1], 
            c=test_output_colors, marker='x', label='output_df_sub', alpha=0.99)

# ChemNet
unique_smiles_chemnet = chemnet_sub['SMILES'].unique()
color_map_chemnet = {smiles: cm.tab20(i % 20) for i, smiles in enumerate(unique_smiles_chemnet)}
chemnet_colors = chemnet_sub['SMILES'].map(color_map_chemnet)
plt.scatter(X_chemnet_pca[:,0], X_chemnet_pca[:,1], 
            c=chemnet_colors, marker='o', label='chemnet_sub', alpha=0.99) 

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA of ChemNet Embeddings (Unique SMILES Colored)')
plt.tight_layout()
plt.show()

In [ ]:
# PCA plot, remove outliers after PCA, color by SMILES

# Prepare data
emb_cols_output = get_embedding_columns(test_output_df_sub)
emb_cols_chemnet = get_embedding_columns(chemnet_sub)

# Get the actual data as numpy arrays
X_test_output = test_output_df_sub[emb_cols_output].values
X_chemnet = chemnet_sub[emb_cols_chemnet].values

# Combine for PCA fit
X_all = np.vstack([X_test_output, X_chemnet])
pca = PCA(n_components=2)
X_all_pca = pca.fit_transform(X_all)

# Re-split
X_test_output_pca = X_all_pca[:len(test_output_df_sub)]
X_chemnet_pca = X_all_pca[len(test_output_df_sub):]

# Combine for easier filtering
test_output_pca_df = test_output_df_sub.copy()
test_output_pca_df['PC1'] = X_test_output_pca[:, 0]
test_output_pca_df['PC2'] = X_test_output_pca[:, 1]

chemnet_pca_df = chemnet_sub.copy()
chemnet_pca_df['PC1'] = X_chemnet_pca[:, 0]
chemnet_pca_df['PC2'] = X_chemnet_pca[:, 1]

# Remove outliers on PC2 
test_output_pca_df = test_output_pca_df.sort_values('PC2', ascending=True)
test_output_pca_df = test_output_pca_df.iloc[:-2]  
chemnet_pca_df = chemnet_pca_df.sort_values('PC2', ascending=True)
chemnet_pca_df = chemnet_pca_df.iloc[:-2]

# Remove outliers on PC1 
test_output_pca_df = test_output_pca_df.sort_values('PC1', ascending=True)
test_output_pca_df = test_output_pca_df.iloc[:-4]   
chemnet_pca_df = chemnet_pca_df.sort_values('PC1', ascending=True)
chemnet_pca_df = chemnet_pca_df.iloc[:-4] 

# Plot
plt.figure(figsize=(10,7))

# Plot points and color by SMILES 
# Ouptut
unique_smiles_test_output = test_output_pca_df['SMILES_spectra'].unique()
color_map_test_output = {smiles: cm.tab20(i % 20) for i, smiles in enumerate(unique_smiles_test_output)}
test_output_colors = test_output_pca_df['SMILES_spectra'].map(color_map_test_output)
plt.scatter(test_output_pca_df['PC1'], test_output_pca_df['PC2'],
            c=test_output_colors, marker='x', label='test_output_df_sub', alpha=0.99)

# ChemNet
unique_smiles_chemnet = chemnet_pca_df['SMILES'].unique()
color_map_chemnet = {smiles: cm.tab20(i % 20) for i, smiles in enumerate(unique_smiles_chemnet)}
chemnet_colors = chemnet_pca_df['SMILES'].map(color_map_chemnet)
plt.scatter(chemnet_pca_df['PC1'], chemnet_pca_df['PC2'],
            c=chemnet_colors, marker='o', label='chemnet_sub', alpha=0.99)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA of ChemNet Embeddings (Selected Outliers Removed, Unique SMILES Colored)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PCA plot, remove ouliers after PCA, color by toxicity

# Prepare data
emb_cols_test_output = get_embedding_columns(test_output_df_sub)
emb_cols_chemnet = get_embedding_columns(chemnet_sub)

# Get the actual data as numpy arrays
X_test_output = test_output_df_sub[emb_cols_test_output].values
X_chemnet = chemnet_sub[emb_cols_chemnet].values

# Combine for PCA fit
X_all = np.vstack([X_test_output, X_chemnet])
pca = PCA(n_components=2)
X_all_pca = pca.fit_transform(X_all)

# Split back
X_test_output_pca = X_all_pca[:len(test_output_df_sub)]
X_chemnet_pca = X_all_pca[len(test_output_df_sub):]

# Build DataFrames for easier filtering
test_output_pca_df = test_output_df_sub.copy()
test_output_pca_df['PC1'] = X_test_output_pca[:, 0]
test_output_pca_df['PC2'] = X_test_output_pca[:, 1]

chemnet_pca_df = chemnet_sub.copy()
chemnet_pca_df['PC1'] = X_chemnet_pca[:, 0]
chemnet_pca_df['PC2'] = X_chemnet_pca[:, 1]

# Remove outliers using the function
test_output_pca_df = remove_outliers_on_axes(test_output_pca_df, 'PC1', 'PC2', n_high_pc1=4, n_high_pc2=2)
chemnet_pca_df = remove_outliers_on_axes(chemnet_pca_df, 'PC1', 'PC2', n_high_pc1=4, n_high_pc2=2)

# Plot
plt.figure(figsize=(8,6))

# Plot points and color by toxicity 
# Output
mask_test_output_red = test_output_pca_df['tox_epa12'] == 1
mask_test_output_blue = test_output_pca_df['tox_epa12'] == 0
plt.scatter(test_output_pca_df.loc[mask_test_output_blue, 'PC1'], test_output_pca_df.loc[mask_test_output_blue, 'PC2'], 
            c='blue', marker='x', label='test_output_df_sub, tox_epa12=0', alpha=0.7)
plt.scatter(test_output_pca_df.loc[mask_test_output_red, 'PC1'], test_output_pca_df.loc[mask_test_output_red, 'PC2'], 
            c='red', marker='x', label='test_output_df_sub, tox_epa12=1', alpha=0.7)

# ChemNet
mask_chemnet_red = chemnet_pca_df['tox_epa12'] == 1
mask_chemnet_blue = chemnet_pca_df['tox_epa12'] == 0
plt.scatter(chemnet_pca_df.loc[mask_chemnet_blue, 'PC1'], chemnet_pca_df.loc[mask_chemnet_blue, 'PC2'], 
            c='blue', marker='o', label='chemnet_sub, tox_epa12=0', alpha=0.7)
plt.scatter(chemnet_pca_df.loc[mask_chemnet_red, 'PC1'], chemnet_pca_df.loc[mask_chemnet_red, 'PC2'], 
            c='red', marker='o', label='chemnet_sub, tox_epa12=1', alpha=0.7)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA of ChemNet Embeddings (Outliers Removed)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PCA plot, remove outliers before PCA, color by toxicity 

# Prepare data
emb_cols_test_output = get_embedding_columns(test_output_df_sub)
emb_cols_chemnet = get_embedding_columns(chemnet_sub)

# Remove outliers
test_output_df_sub_clean = remove_outliers_by_column(test_output_df_sub, emb_cols_test_output[0], n_low=1, n_high=1)
chemnet_sub_clean = remove_outliers_by_column(chemnet_sub, emb_cols_chemnet[0], n_low=1, n_high=1)

# Get the data as numpy arrays
X_test_output = test_output_df_sub_clean[emb_cols_test_output].values
X_chemnet = chemnet_sub_clean[emb_cols_chemnet].values

# Combine for PCA fit
X_all = np.vstack([X_test_output, X_chemnet])
pca = PCA(n_components=2)
X_all_pca = pca.fit_transform(X_all)

# Re-split
X_test_output_pca = X_all_pca[:len(test_output_df_sub_clean)]
X_chemnet_pca = X_all_pca[len(test_output_df_sub_clean):]

# Plot
plt.figure(figsize=(8,6))

# Plot points and color by toxicity 
# Output
mask_test_output_red = test_output_df_sub_clean['tox_epa12'] == 1
mask_test_output_blue = test_output_df_sub_clean['tox_epa12'] == 0
plt.scatter(X_test_output_pca[mask_test_output_blue,0], X_test_output_pca[mask_test_output_blue,1], 
            c='blue', marker='x', label='test_output_df_sub, tox_epa12=0', alpha=0.7)
plt.scatter(X_test_output_pca[mask_test_output_red,0], X_test_output_pca[mask_test_output_red,1], 
            c='red', marker='x', label='test_output_df_sub, tox_epa12=1', alpha=0.7)

# ChemNet
mask_chemnet_red = chemnet_sub_clean['tox_epa12'] == 1
mask_chemnet_blue = chemnet_sub_clean['tox_epa12'] == 0
plt.scatter(X_chemnet_pca[mask_chemnet_blue,0], X_chemnet_pca[mask_chemnet_blue,1], 
            c='blue', marker='o', label='chemnet_sub, tox_epa12=0', alpha=0.7)
plt.scatter(X_chemnet_pca[mask_chemnet_red,0], X_chemnet_pca[mask_chemnet_red,1], 
            c='red', marker='o', label='chemnet_sub, tox_epa12=1', alpha=0.7)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA of ChemNet Embeddings (Outliers Removed Before PCA)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PCA plot, remove outliers before PCA, color by SMILES

# Prepare data
emb_cols_output = get_embedding_columns(test_output_df_sub)
emb_cols_chemnet = get_embedding_columns(chemnet_sub)

# Remove outliers
test_output_df_sub_clean = remove_outliers_by_column(test_output_df_sub, emb_cols_output[0], n_low=1, n_high=1)
chemnet_sub_clean = remove_outliers_by_column(chemnet_sub, emb_cols_chemnet[0], n_low=1, n_high=1)

# Get the actual data as numpy arrays
X_test_output = test_output_df_sub_clean[emb_cols_output].values
X_chemnet = chemnet_sub_clean[emb_cols_chemnet].values

# Combine for PCA fit
X_all = np.vstack([X_test_output, X_chemnet])
pca = PCA(n_components=2)
X_all_pca = pca.fit_transform(X_all)

# Re-split
X_test_output_pca = X_all_pca[:len(test_output_df_sub_clean)]
X_chemnet_pca = X_all_pca[len(test_output_df_sub_clean):]

# Plot
plt.figure(figsize=(10,7))

#  PLot points and color by SMILES 
# Output
unique_smiles_test_output = test_output_df_sub_clean['SMILES_spectra'].unique()
color_map_test_output = {smiles: cm.tab20(i % 20) for i, smiles in enumerate(unique_smiles_test_output)}
test_output_colors = test_output_df_sub_clean['SMILES_spectra'].map(color_map_test_output)
plt.scatter(X_test_output_pca[:,0], X_test_output_pca[:,1], 
            c=test_output_colors, marker='x', label='test_output_df_sub', alpha=0.99)

# ChemNet
unique_smiles_chemnet = chemnet_sub_clean['SMILES'].unique()
color_map_chemnet = {smiles: cm.tab20(i % 20) for i, smiles in enumerate(unique_smiles_chemnet)}
chemnet_colors = chemnet_sub_clean['SMILES'].map(color_map_chemnet)
plt.scatter(X_chemnet_pca[:,0], X_chemnet_pca[:,1], 
            c=chemnet_colors, marker='o', label='chemnet_sub', alpha=0.99)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA of ChemNet Embeddings (Outliers Removed Before PCA, Unique SMILES Colored)')
plt.tight_layout()
plt.show()

### PCA train and test processing

In [ ]:
print(type(train_data_copy))

In [ ]:
# Mapping from SMILES to Response
smiles_to_response = df2_pos.drop_duplicates(subset='SMILES_spectra').set_index('SMILES_spectra')['Response']

# Ouptput dataframe
train_output_np = train_output.cpu().numpy()
train_output_df = pd.DataFrame(train_output_np)
train_output_df['SMILES_spectra'] = train_data_copy['SMILES_spectra'].values  # 1:1 with test_data
cols = train_output_df.columns.tolist()
train_output_df = train_output_df[['SMILES_spectra'] + [c for c in cols if c != 'SMILES_spectra']]
train_output_df['Response'] = train_output_df['SMILES_spectra'].map(smiles_to_response)

# Check the output
train_output_df.head()
print(train_output_df.shape)
print(train_output_df['SMILES_spectra'].value_counts())

In [ ]:
# Find SMILES_spectra with 'occurances' or fewer/more occurrences
occurances = 10 # This can be changed
smiles_counts = test_output_df['SMILES_spectra'].value_counts()
train_smiles_counts = train_output_df['SMILES_spectra'].value_counts()

#fewer_select_smiles = smiles_counts[smiles_counts <= occurances].index
more_select_smiles = train_smiles_counts[train_smiles_counts >= occurances].index
# Subset train_output_df
#train_output_df_sub = train_output_df[train_output_df['SMILES_spectra'].isin(fewer_select_smiles)].copy()
train_output_df_sub = train_output_df[train_output_df['SMILES_spectra'].isin(more_select_smiles)].copy()


print(train_output_df_sub.shape)
train_output_df_sub.head()

num_unique_smiles = train_output_df_sub['SMILES_spectra'].nunique()
print("Number of unique SMILES_spectra:", num_unique_smiles)

#fewer_select_smiles = smiles_counts[smiles_counts <= occurances].index
more_select_smiles = smiles_counts[smiles_counts >= occurances].index
# Subset test_output_df
#test_output_df_sub = test_output_df[test_output_df['SMILES_spectra'].isin(fewer_select_smiles)].copy()
test_output_df_sub = test_output_df[test_output_df['SMILES_spectra'].isin(more_select_smiles)].copy()

print(test_output_df_sub.shape)
test_output_df_sub.head()

num_unique_smiles = test_output_df_sub['SMILES_spectra'].nunique()
print("Number of unique SMILES_spectra:", num_unique_smiles)


In [ ]:
# Upload the Chemnet embeddings dataframe to provide true embeddings 
ChemNet_of_df2pos_no_repeats = pd.read_csv("/home/dlipsey/Research/MITLincolnLabs/MIT_LL_data/ChemNet_of_df2pos_no_repeats.csv")

# Get the unique SMILES_spectra for the subset
sub_smiles_set = set(test_output_df_sub['SMILES_spectra'].unique()) # With a unique dataframe this isn't strictly necessary

# Subset ChemNet_of_df2pos_no_repeats to only those SMILES
chemnet_sub = ChemNet_of_df2pos_no_repeats[ChemNet_of_df2pos_no_repeats['SMILES'].isin(sub_smiles_set)].copy()

# Check
print(chemnet_sub.shape)
chemnet_sub.head()


In [ ]:
# Merge with the Response column from df2 to get toxicity values
chemnet_sub = chemnet_sub.merge(
    df2[['SMILES_spectra', 'Response']],
    left_on='SMILES',
    right_on='SMILES_spectra',
    how='left'
)
# Remove the SMILES_spectra column after merging as it's redundant
chemnet_sub = chemnet_sub.drop(columns=['SMILES_spectra'])

# Keep only one row per unique SMILES (first occurrence)
chemnet_sub= chemnet_sub.drop_duplicates(subset='SMILES').copy()

print(chemnet_sub.shape)
chemnet_sub.head()



In [ ]:
# Create tox_epa12 column based on Response
test_output_df_sub['tox_epa12'] = (test_output_df_sub['Response'] <= 500).astype(int)
train_output_df_sub['tox_epa12'] = (train_output_df_sub['Response'] <= 500).astype(int)
chemnet_sub['tox_epa12'] = (chemnet_sub['Response'] <= 500).astype(int)

# Check the disribution
print(test_output_df_sub['tox_epa12'].value_counts())
print(train_output_df_sub['tox_epa12'].value_counts())
print(chemnet_sub['tox_epa12'].value_counts())

### PCA train and test plotting

In [ ]:
# PCA plot, color by toxicity (with train data plotted)

# Prepare data
emb_cols_test_output = get_embedding_columns(test_output_df_sub)
emb_cols_train_output = get_embedding_columns(train_output_df_sub)
emb_cols_chemnet = get_embedding_columns(chemnet_sub)

# Get the actual data as numpy arrays
X_test_output = test_output_df_sub[emb_cols_test_output].values
X_train_output = train_output_df_sub[emb_cols_train_output].values
X_chemnet = chemnet_sub[emb_cols_chemnet].values

# Combine for PCA fit
X_all = np.vstack([X_test_output, X_train_output, X_chemnet])
pca = PCA(n_components=2)
X_all_pca = pca.fit_transform(X_all)

# Re-split
X_output_pca = X_all_pca[:len(test_output_df_sub)]
X_train_output_pca = X_all_pca[len(test_output_df_sub):len(test_output_df_sub) + len(train_output_df_sub)]
X_chemnet_pca = X_all_pca[len(test_output_df_sub) + len(train_output_df_sub):]

# Plot
plt.figure(figsize=(10,7))

# Output (test) points
mask_test_output_red = test_output_df_sub['tox_epa12'] == 1
mask_test_output_blue = test_output_df_sub['tox_epa12'] == 0
plt.scatter(X_output_pca[mask_test_output_blue,0], X_output_pca[mask_test_output_blue,1], 
            c='blue', marker='x', label='test_output_df_sub, tox_epa12=0', alpha=0.7)
plt.scatter(X_output_pca[mask_test_output_red,0], X_output_pca[mask_test_output_red,1], 
            c='red', marker='x', label='test_output_df_sub, tox_epa12=1', alpha=0.7)

# Train points
mask_train_red = train_output_df_sub['tox_epa12'] == 1
mask_train_blue = train_output_df_sub['tox_epa12'] == 0
plt.scatter(X_train_output_pca[mask_train_blue,0], X_train_output_pca[mask_train_blue,1], 
            c='blue', marker='^', label='train_output_df_sub, tox_epa12=0', alpha=0.7)
plt.scatter(X_train_output_pca[mask_train_red,0], X_train_output_pca[mask_train_red,1], 
            c='red', marker='^', label='train_output_df_sub, tox_epa12=1', alpha=0.7)

# ChemNet points
mask_chemnet_red = chemnet_sub['tox_epa12'] == 1
mask_chemnet_blue = chemnet_sub['tox_epa12'] == 0
plt.scatter(X_chemnet_pca[mask_chemnet_blue,0], X_chemnet_pca[mask_chemnet_blue,1], 
            c='blue', marker='o', label='chemnet_sub, tox_epa12=0', alpha=0.7)
plt.scatter(X_chemnet_pca[mask_chemnet_red,0], X_chemnet_pca[mask_chemnet_red,1], 
            c='red', marker='o', label='chemnet_sub, tox_epa12=1', alpha=0.7)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA of ChemNet Embeddings (Train/Test/ChemNet, colored by toxicity)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PCA plot, color by SMILES (with train data plotted)

# Prepare data
emb_cols_test_output = get_embedding_columns(test_output_df_sub)
emb_cols_train_output = get_embedding_columns(train_output_df_sub)
emb_cols_chemnet = get_embedding_columns(chemnet_sub)

# Get the actual data as numpy arrays
X_test_output = test_output_df_sub[emb_cols_test_output].values
X_train_output = train_output_df_sub[emb_cols_train_output].values
X_chemnet = chemnet_sub[emb_cols_chemnet].values

# Combine for PCA fit
X_all = np.vstack([X_test_output, X_train_output, X_chemnet])
pca = PCA(n_components=2)
X_all_pca = pca.fit_transform(X_all)

# Re-split 
X_test_output_pca = X_all_pca[:len(test_output_df_sub)]
X_train_output_pca = X_all_pca[len(test_output_df_sub):len(test_output_df_sub) + len(train_output_df_sub)]
X_chemnet_pca = X_all_pca[len(test_output_df_sub) + len(train_output_df_sub):]

# Assign colors to unique SMILES
all_smiles = pd.concat([
    test_output_df_sub['SMILES_spectra'],
    train_output_df_sub['SMILES_spectra'],
    chemnet_sub['SMILES']
]).unique()
color_map = {smiles: cm.tab20(i % 20) for i, smiles in enumerate(all_smiles)}

output_colors = test_output_df_sub['SMILES_spectra'].map(color_map)
train_colors = train_output_df_sub['SMILES_spectra'].map(color_map)
chemnet_colors = chemnet_sub['SMILES'].map(color_map)

# Plot
plt.figure(figsize=(10,7))

plt.scatter(X_output_pca[:,0], X_output_pca[:,1], 
            c=output_colors, marker='x', label='test_output_df_sub', alpha=0.99)
plt.scatter(X_train_output_pca[:,0], X_train_output_pca[:,1], 
            c=train_colors, marker='^', label='train_output_df_sub', alpha=0.99)
plt.scatter(X_chemnet_pca[:,0], X_chemnet_pca[:,1], 
            c=chemnet_colors, marker='o', label='chemnet_sub', alpha=0.99)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA of ChemNet Embeddings (Train/Test/ChemNet, colored by SMILES)')
plt.tight_layout()
plt.legend()
plt.show()

In [ ]:
# PCA plot, color by SMILES (with train data plotted and SMILES legend)

import matplotlib.patches as mpatches

# Prepare data
emb_cols_test_output = get_embedding_columns(test_output_df_sub)
emb_cols_train_output = get_embedding_columns(train_output_df_sub)
emb_cols_chemnet = get_embedding_columns(chemnet_sub)

# Get the actual data as numpy arrays
X_test_output = test_output_df_sub[emb_cols_test_output].values
X_train_output = train_output_df_sub[emb_cols_train_output].values
X_chemnet = chemnet_sub[emb_cols_chemnet].values

# Combine for PCA fit
X_all = np.vstack([X_test_output, X_train_output, X_chemnet])
pca = PCA(n_components=2)
X_all_pca = pca.fit_transform(X_all)

# Re-split
X_test_output_pca = X_all_pca[:len(test_output_df_sub)]
X_train_output_pca = X_all_pca[len(test_output_df_sub):len(test_output_df_sub) + len(train_output_df_sub)]
X_chemnet_pca = X_all_pca[len(test_output_df_sub) + len(train_output_df_sub):]

# Assign colors to unique SMILES
all_smiles = pd.concat([
    test_output_df_sub['SMILES_spectra'],
    train_output_df_sub['SMILES_spectra'],
    chemnet_sub['SMILES']
]).unique()
color_map = {smiles: cm.tab20(i % 20) for i, smiles in enumerate(all_smiles)}

test_output_colors = test_output_df_sub['SMILES_spectra'].map(color_map)
train_colors = train_output_df_sub['SMILES_spectra'].map(color_map)
chemnet_colors = chemnet_sub['SMILES'].map(color_map)

plt.figure(figsize=(12,8))

plt.scatter(X_test_output_pca[:,0], X_test_output_pca[:,1], 
            c=test_output_colors, marker='x', label='test_output_df_sub', alpha=0.99)
plt.scatter(X_train_output_pca[:,0], X_train_output_pca[:,1], 
            c=train_colors, marker='^', label='train_output_df_sub', alpha=0.99)
plt.scatter(X_chemnet_pca[:,0], X_chemnet_pca[:,1], 
            c=chemnet_colors, marker='o', label='chemnet_sub', alpha=0.99)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA of ChemNet Embeddings (Train/Test/ChemNet, colored by SMILES)')

# First legend: dataset markers (handles only)
handles, labels = plt.gca().get_legend_handles_labels()
legend1 = plt.legend(handles, labels, loc='upper right', title='Dataset')

# Second legend: SMILES color mapping (show up to 20 for clarity)
max_smiles_legend = 20
smiles_legend_patches = [
    mpatches.Patch(color=color_map[smiles], label=smiles)
    for i, smiles in enumerate(all_smiles[:max_smiles_legend])
]
legend2 = plt.legend(handles=smiles_legend_patches, title="SMILES", loc='center left', bbox_to_anchor=(1.01, 0.5))
plt.gca().add_artist(legend1)  # Add the first legend back

plt.tight_layout()
plt.show()

In [ ]:
print(chemnet_sub['SMILES'])
# The tan molecule is: 2-(diethylamino)-N-(2,6-dimethylphenyl)acetamide
# The dark green molecule is: 4-amino-5-chloro-N-[2-(diethylamino)ethyl]-2-methoxybenzamide